# Road Network Extraction and Simplification

[![image](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/opengeos/geoai/blob/main/docs/examples/road_network.ipynb)

This notebook demonstrates how to extract road networks from deep learning segmentation masks and simplify them using the [neatnet](https://github.com/uscuni/neatnet) library.

## Workflow

1. **Skeletonize** a binary road mask to extract centerlines
2. **Vectorize** the skeleton into LineString geometries
3. **Simplify** the network with neatnet (removes dual carriageways, roundabouts, complex intersections)

## Install packages

Uncomment the following line to install the required packages.

In [ ]:
# %pip install geoai-py[road]

## Import libraries

In [ ]:
import geoai

## Create a synthetic road mask

For demonstration purposes, we create a synthetic road mask raster with a grid pattern of roads. In practice, this would be the output of a deep learning road segmentation model.

In [ ]:
import numpy as np
import rasterio
from rasterio.transform import from_bounds

width, height = 500, 500
data = np.zeros((1, height, width), dtype=np.uint8)

# Create a grid of roads (3-pixel wide)
for y in range(100, height, 100):
    data[0, y - 1 : y + 2, :] = 1  # horizontal roads
for x in range(100, width, 100):
    data[0, :, x - 1 : x + 2] = 1  # vertical roads

# Add a diagonal road
for i in range(50, 400):
    j = int(50 + i * 0.8)
    if j < width:
        data[0, max(0, i - 1) : i + 2, max(0, j - 1) : j + 2] = 1

mask_path = "road_mask_demo.tif"
transform = from_bounds(500000, 4500000, 500500, 4500500, width, height)

with rasterio.open(
    mask_path,
    "w",
    driver="GTiff",
    height=height,
    width=width,
    count=1,
    dtype=np.uint8,
    crs="EPSG:32618",
    transform=transform,
) as dst:
    dst.write(data)

print(f"Created road mask: {mask_path} ({width}x{height} pixels)")

Created road mask: road_mask_demo.tif (500x500 pixels)


## Visualize input mask

View the binary road mask. White pixels (value 1) represent roads.

In [ ]:
geoai.view_raster(mask_path)

## Extract road network (without neatnet)

First, extract the raw road network by skeletonizing the mask and vectorizing it. Set `neatify=False` to skip the neatnet simplification step.

In [ ]:
raw_roads = geoai.extract_road_network(
    mask_path,
    neatify=False,
    min_length=5.0,
)
print(f"Extracted {len(raw_roads)} road segments")
print(f"Total network length: {raw_roads.geometry.length.sum():.1f} m")

Extracted 57 road segments
Total network length: 4318.5 m


### Visualize raw extracted lines

In [ ]:
geoai.view_vector_interactive(raw_roads)

## Extract road network (with neatnet)

Now extract the road network with neatnet simplification enabled. This removes dual carriageways, roundabouts, and complex intersections, replacing them with clean centerlines.

In [ ]:
cleaned_roads = geoai.extract_road_network(
    mask_path,
    output_path="roads_cleaned.gpkg",
    neatify=True,
    min_length=5.0,
)
print(f"Cleaned road segments: {len(cleaned_roads)}")
print(f"Total network length: {cleaned_roads.geometry.length.sum():.1f} m")

ImportError: neatnet is required for network topology operations. Install it with: pip install neatnet
Or install geoai with network support: pip install geoai-py[network]
Note: neatnet requires Python >= 3.11.

### Visualize cleaned network

In [ ]:
geoai.view_vector_interactive(cleaned_roads)

## Compare raw vs. cleaned networks

Use a split map to compare the raw extracted network with the neatnet-simplified version.

In [ ]:
geoai.create_split_map(
    left_layer=raw_roads,
    right_layer=cleaned_roads,
    left_args={"style": {"color": "red", "weight": 2}},
    right_args={"style": {"color": "blue", "weight": 2}},
)

## Network statistics

Compare key metrics between the raw and cleaned networks.

In [ ]:
import pandas as pd

stats = pd.DataFrame(
    {
        "Raw": [
            len(raw_roads),
            f"{raw_roads.geometry.length.sum():.1f}",
            f"{raw_roads.geometry.length.mean():.1f}",
        ],
        "Cleaned (neatnet)": [
            len(cleaned_roads),
            f"{cleaned_roads.geometry.length.sum():.1f}",
            f"{cleaned_roads.geometry.length.mean():.1f}",
        ],
    },
    index=["Segments", "Total length (m)", "Mean segment length (m)"],
)
stats

## Using the low-level API

You can also use the lower-level `raster_mask_to_linestrings()` and `neatify_network()` functions separately for more control.

In [ ]:
from geoai.utils.raster import raster_mask_to_linestrings
from geoai.road import neatify_network

# Step 1: Skeletonize and vectorize
lines = raster_mask_to_linestrings(
    mask_path,
    threshold=0,
    min_length=5.0,
    simplify_tolerance=1.0,
)
print(f"Vectorized {len(lines)} line segments")

# Step 2: Simplify with neatnet
simplified = neatify_network(lines)
print(f"Simplified to {len(simplified)} segments")

## Summary

This notebook demonstrated:

1. **Road mask to vector** — converting binary road masks to LineString geometries via skeletonization
2. **Network simplification** — using neatnet to clean up dual carriageways, roundabouts, and complex intersections
3. **Comparison** — visualizing raw vs. cleaned networks side by side
4. **Low-level API** — using `raster_mask_to_linestrings()` and `neatify_network()` for fine-grained control

### Key Functions

| Function | Description |
|----------|-------------|
| `geoai.extract_road_network()` | End-to-end road extraction: mask → skeleton → lines → neatnet |
| `geoai.neatify_network()` | Simplify an existing road network GeoDataFrame with neatnet |
| `geoai.utils.raster.raster_mask_to_linestrings()` | Convert binary mask to LineStrings via skeletonization |

### References

- neatnet: https://github.com/uscuni/neatnet
- scikit-image skeletonize: https://scikit-image.org/docs/stable/api/skimage.morphology.html#skimage.morphology.skeletonize